# A股主板双模式日线筛选器
收盘后或次日开盘前运行。代码从 GitHub 获取，缓存、外部证据、状态和每日结果保存在 Google Drive。

In [ ]:
REPO_URL = "https://github.com/YOUR_NAME/ashare-daily-scanner.git"
BRANCH = "main"
CONFIG_PATH = "config/default.yaml"  # 高召回对照可改为 config/high_recall.yaml
DRIVE_DATA_DIR = "/content/drive/MyDrive/ashare-daily-scanner-data"
RUN_BACKTEST = False
BACKTEST_START = "2024-01-01"
BACKTEST_END = "2025-12-31"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pathlib
import subprocess
import sys

if 'YOUR_NAME' in REPO_URL:
    raise ValueError('иЇ·е…€жЉЉ REPO_URL ж”№ж€ђдЅ зљ„ GitHub д»“еє“ењ°еќЂ')

repo_dir = pathlib.Path('/content/ashare-daily-scanner')
if (repo_dir / '.git').exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(repo_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_dir)], check=True)
os.chdir(repo_dir)
print(repo_dir)

In [ ]:
command = [
    sys.executable, '-m', 'ashare_scanner',
    '--config', CONFIG_PATH,
    '--data-dir', DRIVE_DATA_DIR,
    'run',
]
subprocess.run(command, check=True)

In [ ]:
import json
import pandas as pd
from IPython.display import display

latest = json.loads((pathlib.Path(DRIVE_DATA_DIR) / 'latest_run.json').read_text(encoding='utf-8'))
run_dir = pathlib.Path(latest['run_dir'])
report = json.loads((run_dir / 'coverage_report.json').read_text(encoding='utf-8'))
print(json.dumps(report['signals'], ensure_ascii=False, indent=2))
display(pd.read_csv(run_dir / 'watchlist_active.csv', dtype={'code': str}).head(100))

In [ ]:
import json
import pandas as pd
from IPython.display import display

latest = json.loads((pathlib.Path(DRIVE_DATA_DIR) / 'latest_run.json').read_text(encoding='utf-8'))
run_dir = pathlib.Path(latest['run_dir'])
report = json.loads((run_dir / 'coverage_report.json').read_text(encoding='utf-8'))
print('信号数量:', json.dumps(report['signals'], ensure_ascii=False))
print('诊断:', report['screening']['assessment'])
print('外部证据:', report.get('external_evidence', {}))

labels = {
    'accumulation_late': '强资金运作型（埋伏吸筹末期）',
    'main_wave': '强资金运作型（主升浪阶段）',
}
for signal, label in labels.items():
    result = pd.read_csv(run_dir / f'{signal}_all.csv', dtype={'code': str})
    print(f'\\n[{label}] 共 {len(result)} 只')
    if result.empty:
        print('无符合条件股票；CSV 只有表头属于正常结果。')
    else:
        columns = [c for c in ['rank', 'code', 'name', 'close', 'pct_chg', 'accumulation_score', 'main_wave_score', 'accumulation_evidence_groups', 'rs20_percentile', 'vol_ratio_20', 'turnover_ratio_20', 'main_net_inflow_ratio_pct'] if c in result.columns]
        display(result[columns].head(100))

print('\\n当前观察池')
display(pd.read_csv(run_dir / 'watchlist_active.csv', dtype={'code': str}).head(100))